In [2]:
import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import torch.optim as optim

from sklearn.metrics import accuracy_score, classification_report

# Load and pre process

In [3]:
# Transform image: Min max scaling (0, 1), Normalize (-1, 1)
transform = transforms.Compose(
    [
        # data augmentaiton 
        transforms.RandomHorizontalFlip(), # flip horizontally
        transforms.ColorJitter(brightness=0.2), # 
        transforms.RandomResizedCrop(size=32, scale=(0.8, 1.0), antialias=True), # randomly crop keeping 80-100%

        transforms.ToTensor(), # convert to tensore and min max scale
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # Normalize (-1, 1)
    ]
)

In [4]:
# Loading dataset
trainset = CIFAR10(
    root="datasets/cnn_data", 
    # for first time it was True
    download=False,
    transform=transform,
    train=True
)
testset = CIFAR10(
    root="datasets/cnn_data",
    transform=transform,
    train=False
)

In [5]:
trainset, valset = random_split(
    dataset=trainset,
    lengths=[0.8, 0.2]
)

In [6]:
# Convert to DataLoader
# CIFAR10 already returned the datasets
train_loader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    valset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    testset,
    batch_size=64,
    shuffle=True
)

# Build CNN

In [17]:
# # Define the model
# class CNN(nn.Module):

#     def __init__(self):
#         super().__init__()

#         self.conv_layers = nn.Sequential(
#             # Image shape: 32 x 32 x 3
#             nn.Conv2d(3, 32, kernel_size=3, padding=1), # 32 x 32 x 32
#             nn.ReLU(), 
#             nn.MaxPool2d(kernel_size=2, stride=2), # 16 x 16 x 32

#             nn.Conv2d(32, 64, kernel_size=3, padding=1), # 16 x 16 x 64
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2), # 8 x 8 x 64

#             nn.Conv2d(64, 128, kernel_size=3, padding=1), # 8 x 8 x 128
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2), # 4 x 4 x 128
#         )

#         self.fc_layers = nn.Sequential(
#             nn.Linear(4*4*128, 256),
#             nn.ReLU(),

#             nn.Linear(256, 10),
#         ) 

#     def forward(self, x):
#         x = self.conv_layers(x)
#         x = x.view(x.size(0), -1) # flattent 4 x 4 x 128 to 1d
#         x = self.fc_layers(x)
#         return x
        
# Define the model
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            # Image shape: 32 x 32 x 3 
            nn.Conv2d(3, 8, kernel_size=3, padding=1), # 32 x 32 x 8
            nn.ReLU(), 

            nn.Conv2d(8, 16, kernel_size=3, padding=1), # 32 x 32 x 16
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 16 x 16 x 16

            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 16 x 16 x 32
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 16 x 16 x 64
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 8 x 8 x 64

            nn.Conv2d(64, 64, kernel_size=3, padding=1), # 8 x 8 x 64
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 8 x 8 x 128
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 4 x 4 x 128

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10),
        ) 

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattened 4 x 4 x 128 to 1d
        x = self.fc_layers(x)
        return x
        

In [18]:
# Build the model
cnn_model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(cnn_model.parameters())

In [19]:
# Training
epochs = 10
min_val_loss = float("inf")

for epoch in range(0, epochs):
    # training
    cnn_model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = cnn_model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    # validation
    cnn_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            y_pred = cnn_model.forward(images)
            loss = criterion(y_pred, labels)

            val_loss += loss.item()

    val_loss /= len(test_loader)
    if (val_loss < min_val_loss):
        min_val_loss = val_loss
        torch.save(cnn_model.state_dict(), "cifar_10_cnn.pt")


    print(f"Epoch: {epoch} Train => {train_loss} Val => {val_loss}")

Epoch: 0 Train => 1.7224045501708984 Val => 1.5318111462198245
Epoch: 1 Train => 1.3462765645027162 Val => 1.2726530557984759
Epoch: 2 Train => 1.161694225215912 Val => 1.1072200407647783
Epoch: 3 Train => 1.0408601477622985 Val => 1.0587225943613963
Epoch: 4 Train => 0.9449400886535645 Val => 0.933571565682721
Epoch: 5 Train => 0.8660290315151215 Val => 0.8818067585586742
Epoch: 6 Train => 0.8069410952091217 Val => 0.867397204705864
Epoch: 7 Train => 0.7628910144329071 Val => 0.8244140592350322
Epoch: 8 Train => 0.7195855471611023 Val => 0.7679487493387454
Epoch: 9 Train => 0.6744028210163117 Val => 0.7705665961572319


In [20]:
# Loading the best model
cnn_model.load_state_dict(torch.load("cifar_10_cnn.pt"))

C:\Users\User\AppData\Local\Temp\ipykernel_26636\1022153840.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("cifar_10_cnn.pt"))


<All keys matched successfully>

In [21]:
# Evaluation
cnn_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        y_pred_probs = cnn_model(xb)
        _, y_pred_labels = torch.max(y_pred_probs, 1)

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())


In [22]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(f"Classification report: {classification_report(y_actual, y_pred)}")

Accuracy: 0.7314
Classification report:               precision    recall  f1-score   support

           0       0.74      0.77      0.76      1000
           1       0.85      0.87      0.86      1000
           2       0.61      0.67      0.64      1000
           3       0.55      0.51      0.53      1000
           4       0.70      0.64      0.67      1000
           5       0.69      0.60      0.64      1000
           6       0.73      0.83      0.78      1000
           7       0.76      0.78      0.77      1000
           8       0.85      0.82      0.83      1000
           9       0.83      0.83      0.83      1000

    accuracy                           0.73     10000
   macro avg       0.73      0.73      0.73     10000
weighted avg       0.73      0.73      0.73     10000

